In [ ]:
import pandas as pd

metadata = pd.read_csv('new_data/Metadata.csv', sep=';')
metadata = metadata.drop(columns=['№', 'Автор', 'Файл_№', 'Начало (сек)', 'Конец (сек)', 'Code'])
metadata = metadata.rename(columns={'Файл (путь от рабочей папки)': 'path', 'Текст': 'text'})
metadata = metadata.dropna()
metadata = metadata.drop_duplicates()
metadata

,path,text
0,Динисламова Светлана Силеверстовна/mns_2efb21c...,Ма̄ньщи ща̄й ва̄рнэ хо̄тпат.
1,Динисламова Светлана Силеверстовна/mns_2efb21c...,Ты йист Саранпавылт о̄лнэ ма̄ньлат ма̄хум пусс...
2,Динисламова Светлана Силеверстовна/mns_2efb21c...,"Матыр йильпи тэ̄ла номылматэ̄гыт, тувыл э̄лал..."
3,Динисламова Светлана Силеверстовна/mns_2efb21c...,Ань «Э̄рупса» нампа община урыл потыр хунтлэн.
4,Динисламова Светлана Силеверстовна/mns_2efb21c...,"Ловмантем та̄л ювле хультум порат, тит сотыра ..."
...,...,...
15598,Теткина Людмила Ильинична/mns_e7a9bbe0-00fc-46...,"Ань ла̄тӈув ос пēс наканув вос ва̄ганыл, вос х..."
15599,Теткина Людмила Ильинична/mns_e7a9bbe0-00fc-46...,"Ма̄н, «Лӯима̄ сэ̄рипос» газета ва̄рнэ ма̄хум, ..."
15600,Теткина Людмила Ильинична/mns_e7a9bbe0-00fc-46...,"Са̄в ёмас, сымыӈ ла̄тӈыл кēтыянӯв!"
15601,Теткина Людмила Ильинична/mns_e7a9bbe0-00fc-46...,"Пус ка̄т, пус ла̄гыл вос о̄лэ̄гыт, яласэ̄гыт, ..."


In [17]:
import uuid
import shutil

dataset = pd.DataFrame(columns=['audio_file', 'text'])

for i in range(0, len(metadata)):
    sample = metadata.iloc[i]

    source_path = 'new_data/' + sample['path']
    audio_file = str(uuid.uuid4()) + '.wav'
    destination_path = 'bonus_dataset/wavs/' + audio_file

    new_row = pd.DataFrame([{'audio_file': audio_file, 'text': sample['text']}])
    dataset = pd.concat([dataset, new_row], ignore_index=True)

    shutil.copy(source_path, destination_path)

dataset

,audio_file,text
0,4cd9fa95-f957-4eea-9d82-97b7ebebdd73.wav,Ма̄ньщи ща̄й ва̄рнэ хо̄тпат.
1,0f86468b-2f60-4e97-b302-f3a101032612.wav,Ты йист Саранпавылт о̄лнэ ма̄ньлат ма̄хум пусс...
2,3a480e4c-ba39-47b1-bacf-b0bb1eefcedb.wav,"Матыр йильпи тэ̄ла номылматэ̄гыт, тувыл э̄лал..."
3,dc7d9da1-358c-4548-8d6a-87863c05e877.wav,Ань «Э̄рупса» нампа община урыл потыр хунтлэн.
4,12d8b344-5892-4505-95d2-7a5b213faa34.wav,"Ловмантем та̄л ювле хультум порат, тит сотыра ..."
...,...,...
15571,9633fde5-5169-47f4-9d78-dd2e37e39e7e.wav,"Ань ла̄тӈув ос пēс наканув вос ва̄ганыл, вос х..."
15572,a4b97cc2-57a6-48d2-b466-7573c81f4c74.wav,"Ма̄н, «Лӯима̄ сэ̄рипос» газета ва̄рнэ ма̄хум, ..."
15573,43946140-52a3-47d1-83d6-525dbfb17568.wav,"Са̄в ёмас, сымыӈ ла̄тӈыл кēтыянӯв!"
15574,234f4c1d-db06-4db0-9c02-9fbd50444db0.wav,"Пус ка̄т, пус ла̄гыл вос о̄лэ̄гыт, яласэ̄гыт, ..."


In [18]:
dataset.to_csv('bonus_dataset/metadata.csv', index=False, sep="\t")

In [19]:
import pandas as pd
import regex as re
import os
import unicodedata

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

df = pd.read_csv('bonus_dataset/metadata.csv', delimiter="\t")
# df

In [20]:
folder_path = 'bonus_dataset/wavs'
file_count = sum(1 for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f)))
print(f"Количество файлов в папке 'wavs': {file_count}")
print(f"Количество записей в df: {len(df)}")

Количество файлов в папке 'wavs': 15576
Количество записей в df: 15576


In [21]:
wavs_list = os.listdir(folder_path)

to_drop = set()
for idx, row in df.iterrows():
    if row['audio_file'] not in wavs_list:
        to_drop.add(idx)

lat_to_cyr = {
    "a": "а", "A": "А",
    "e": "е", "E": "Е",
    "i": "и", "I": "И",
    "o": "о", "O": "О",
    "u": "у", "U": "У",
    "y": "ы", "Y": "Ы",
}

punct_map = {
        '“': '"', '”': '"', '„': '"', '«': '"', '»': '"',
        '‘': "'", '’': "'", '‚': "'",
        '—': '-', '–': '-', '−': '-',
    }

def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFD", str(text))  # 1. разложение
    
    for lat, cyr in lat_to_cyr.items():
        text = text.replace(lat, cyr)
    
    text = unicodedata.normalize("NFC", text)

    text = ''.join(ch for ch in text if unicodedata.category(ch)[0] != 'C')

    text = re.sub(r'\s+', ' ', text)
    
    text = text.strip()
    
    for k, v in punct_map.items():
        text = text.replace(k, v)
    
    return text
    
df['text'] = df['text'].apply(normalize_text)

In [22]:
df = df.drop(to_drop)

df = df.drop_duplicates(subset=['text'])
df = df.dropna()

print(f"Количество записей в df: {len(df)}")

Количество записей в df: 15433


In [23]:
wav_to_drop = []

audio_list = df['audio_file'].tolist()
for wav in wavs_list:
    if wav not in audio_list:
        wav_to_drop.append(wav)

print(f"Количество файлов в папке 'wavs': {len(wavs_list) - len(wav_to_drop)}")

Количество файлов в папке 'wavs': 15433


In [24]:
folder_path = 'bonus_dataset/wavs'

for wav in wav_to_drop:
    file_path = os.path.join(folder_path, wav)
    if os.path.isfile(file_path):
        os.remove(file_path)

print(f"{len(wav_to_drop)} файлов удалено")

df.to_csv('bonus_dataset/metadata.csv', index=False, sep="\t")

143 файлов удалено
